<a href="https://colab.research.google.com/github/Riddha-Rik/Detecting-Demographic-Biases-in-AI-Chatbot-Interactions/blob/feature%2Fbias-auditor-v3/AI_Bias_Auditor_v5_Education_Audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import statsmodels.api as sm
import plotly.express as px

# 1. LOAD THE ORGANIC DATA
# Ensure 'ai_hiring_audit_dataset.csv' is uploaded to your sidebar
df = pd.read_csv('ai_hiring_audit_dataset.csv')
df.columns = df.columns.str.strip()

print(f"✅ Loaded {len(df)} candidate records.")

# 2. PREP THE ENGINE
# We create 'dummy variables' for Job Category so the math can understand text
df_encoded = pd.get_dummies(df, columns=['Job_Category', 'Education_Level'], drop_first=True)

# 3. THE "MERIT VS. BIAS" TEST
# We check if AI_Score is influenced by Years_Experience (Proxy for Age)
# even when we account for their Skill_Fit_Score (Merit)
y = df_encoded['AI_Score']
X = df_encoded[['Years_Experience', 'Skill_Fit_Score']] # Merit and Age Proxy
X = sm.add_constant(X)

# Run the Scientific Audit
audit_model = sm.OLS(y, X.astype(float)).fit()

# 4. GENERATE THE AUDIT REPORT
print("\n" + "="*45)
print("🛡️  OFFICIAL AI BIAS AUDIT: ORGANIC HIRING DATA")
print("="*45)

age_impact = audit_model.params['Years_Experience']
p_value = audit_model.pvalues['Years_Experience']

print(f"Age (Experience) Coefficient: {age_impact:.4f}")
print(f"Statistical Significance (P-Value): {p_value:.4f}")

if p_value < 0.05:
    if age_impact < 0:
        print("\n🚨 BIAS DETECTED: The AI significantly penalizes older/experienced candidates.")
    else:
        print("\n🚨 BIAS DETECTED: The AI significantly favors older/experienced candidates.")
else:
    print("\n✅ PASS: No statistically significant age bias found in this AI model.")

# 5. VISUALIZE THE DISPARITY
fig = px.scatter(df, x="Years_Experience", y="AI_Score",
                 color="Job_Category", trendline="ols",
                 title="Organic Audit: Impact of Experience on AI Hiring Scores",
                 labels={"Years_Experience": "Years of Experience (Age Proxy)", "AI_Score": "AI Rating"},
                 template="plotly_white")

fig.show()

# 6. VIEW SCIENTIFIC SUMMARY
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print("\n--- Full Statistical Summary ---")
    print(audit_model.summary().tables[1])

✅ Loaded 1500 candidate records.

🛡️  OFFICIAL AI BIAS AUDIT: ORGANIC HIRING DATA
Age (Experience) Coefficient: -0.0311
Statistical Significance (P-Value): 0.5430

✅ PASS: No statistically significant age bias found in this AI model.



--- Full Statistical Summary ---
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -4.6107      0.483     -9.545      0.000      -5.558      -3.663
Years_Experience    -0.0311      0.051     -0.608      0.543      -0.131       0.069
Skill_Fit_Score      0.9834      0.016     62.882      0.000       0.953       1.014


In [2]:
import pandas as pd
import statsmodels.api as sm
import plotly.express as px

# 1. Load Data
df = pd.read_csv('ai_hiring_audit_dataset.csv')

# 2. Convert Education into numbers (Bachelors vs Masters vs PhD)
# We use 'get_dummies' to see how much 'extra credit' each degree gets
df_edu = pd.get_dummies(df, columns=['Education_Level'], drop_first=True)

# 3. The "Credentialism" Engine
# We test if Education affects the score, holding Skills constant
y = df_edu['AI_Score']
# We look for the Education columns created by get_dummies
edu_cols = [col for col in df_edu.columns if 'Education_Level' in col]
X = df_edu[['Skill_Fit_Score'] + edu_cols]
X = sm.add_constant(X)

edu_model = sm.OLS(y, X.astype(float)).fit()

# 4. The "Divergence" Plot
# Let's see where Humans and AI disagree the most
fig = px.box(df, x="Education_Level", y="Score_Divergence",
             color="Education_Level", points="all",
             title="Where do Humans and AI Disagree?",
             labels={"Score_Divergence": "Human Score minus AI Score"})

# 5. Output Results
print("🛡️  EDUCATION BIAS AUDIT")
print("="*30)
print(edu_model.summary().tables[1])
fig.show()

🛡️  EDUCATION BIAS AUDIT
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const                        -3.6816      0.689     -5.344      0.000      -5.033      -2.330
Skill_Fit_Score               0.9784      0.011     88.443      0.000       0.957       1.000
Education_Level_Bachelors    -1.3476      0.693     -1.945      0.052      -2.707       0.012
Education_Level_Masters      -0.8407      0.743     -1.132      0.258      -2.298       0.617
Education_Level_PhD          -1.1693      0.930     -1.257      0.209      -2.993       0.655


In [3]:
import pandas as pd
import statsmodels.api as sm
import plotly.express as px

# 1. Load Data
df = pd.read_csv('ai_hiring_audit_dataset.csv')

# 2. Prep for Logistic Regression (Auditing the 'AI_Decision' 0 or 1)
df_dec = pd.get_dummies(df, columns=['Education_Level'], drop_first=True)

# Define X (What the AI sees) and y (The Hire/No-Hire Decision)
y = df_dec['AI_Decision']
X = df_dec[['Skill_Fit_Score', 'Years_Experience'] + [col for col in df_dec.columns if 'Education_Level' in col]]
X = sm.add_constant(X)

# 3. Run Logistic Regression
# We use Logit because the outcome is binary (Yes/No)
logit_model = sm.Logit(y, X.astype(float)).fit()

# 4. The "Human-AI Gap" Visual
# We want to see if Humans are 'kinder' to experienced workers than the AI is
fig = px.scatter(df, x="Years_Experience", y="Score_Divergence",
                 color="Job_Category", trendline="lowess",
                 title="The Fairness Gap: Human Scores vs. AI Scores",
                 labels={"Score_Divergence": "Human Favoritism (Human Score - AI Score)"})

# 5. Output Results
print("\n🛡️  DECISION BIAS AUDIT (LOGISTIC REGRESSION)")
print("="*50)
print(logit_model.summary().tables[1])
fig.show()

Optimization terminated successfully.
         Current function value: 0.128558
         Iterations 10

🛡️  DECISION BIAS AUDIT (LOGISTIC REGRESSION)
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const                       -16.9458      1.557    -10.886      0.000     -19.997     -13.895
Skill_Fit_Score               0.2478      0.022     11.430      0.000       0.205       0.290
Years_Experience             -0.0249      0.042     -0.590      0.555      -0.107       0.058
Education_Level_Bachelors    -0.0108      0.966     -0.011      0.991      -1.904       1.883
Education_Level_Masters      -0.3458      0.976     -0.354      0.723      -2.258       1.567
Education_Level_PhD          -0.0398      1.015     -0.039      0.969      -2.029       1.949


In [4]:
import pandas as pd
import statsmodels.api as sm
import plotly.express as px

# 1. Load Data
df = pd.read_csv('ai_hiring_audit_dataset.csv')

# 2. Create the Interaction Term
# This math checks if (Experience * Education) creates a hidden bias
# e.g., Does the AI penalize PhDs specifically when they have high experience?
df['Exp_PhD_Interaction'] = df['Years_Experience'] * (df['Education_Level'] == 'PhD').astype(int)

# 3. The Intersectional Engine
y = df['AI_Score']
X = df[['Skill_Fit_Score', 'Years_Experience', 'Exp_PhD_Interaction']]
X = sm.add_constant(X)

stress_test = sm.OLS(y, X.astype(float)).fit()

# 4. Visualizing Intersectional Trends
fig = px.scatter(df, x="Years_Experience", y="AI_Score",
                 color="Education_Level", trendline="ols",
                 title="Intersectional Audit: Does Experience impact Degrees differently?",
                 labels={"AI_Score": "AI Hiring Score"})

# 5. Output
print("🛡️  INTERSECTIONAL STRESS TEST")
print("="*30)
print(stress_test.summary().tables[1])
fig.show()

🛡️  INTERSECTIONAL STRESS TEST
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -4.6086      0.496     -9.286      0.000      -5.582      -3.635
Skill_Fit_Score         0.9833      0.017     59.176      0.000       0.951       1.016
Years_Experience       -0.0310      0.052     -0.600      0.548      -0.132       0.070
Exp_PhD_Interaction     0.0011      0.057      0.019      0.985      -0.111       0.113


In [6]:
# FINAL SUMMARY METRICS
total_candidates = len(df)
overall_agreement = df['Decision_Agreement'].mean() * 100
avg_divergence = df['Score_Divergence'].abs().mean()

print("🏁 AUDIT COMPLETION SUMMARY")
print("-" * 30)
print(f"Total Candidates Audited: {total_candidates}")
print(f"Overall Human-AI Agreement: {overall_agreement:.2f}%")
print(f"Average Scoring Friction: {avg_divergence:.2f} points")
print(f"Statistical Status: STABLE / NO SYSTEMIC BIAS DETECTED")

🏁 AUDIT COMPLETION SUMMARY
------------------------------
Total Candidates Audited: 1500
Overall Human-AI Agreement: 90.60%
Average Scoring Friction: 11.49 points
Statistical Status: STABLE / NO SYSTEMIC BIAS DETECTED


In [7]:
# EXPORT THE AUDIT FINDINGS
report_data = {
    'Metric': ['Total Candidates', 'Human-AI Agreement', 'Avg Scoring Friction', 'Age Bias (p-value)', 'Education Bias (p-value)'],
    'Value': [len(df), f"{overall_agreement:.2f}%", f"{avg_divergence:.2f}", f"{p_value:.4f}", f"{edu_model.pvalues.min():.4f}"],
    'Status': ['Complete', 'High', 'Acceptable', 'PASS', 'PASS']
}

report_df = pd.DataFrame(report_data)
report_df.to_csv('AI_Fairness_Audit_Report.csv', index=False)
print("📁 Report saved as 'AI_Fairness_Audit_Report.csv'. You can download this from the sidebar!")

📁 Report saved as 'AI_Fairness_Audit_Report.csv'. You can download this from the sidebar!
